<h1><span style="color:red">Descriptive Statistics for Numeric Variables</span></h1>

This sample notebook reads numeric variables from survey dataset and computes several descriptive statistics

## 1. Retrieve survey parameters from the URL

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Run the credentials cell above, then re-run this cell.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
# Variables used throughout this notebook
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

# NFS image paths (non-empty only when running on a hub with NFS storage)
localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images   # alias used by some notebooks

# Fetch and cache survey CSV for panellibs.extract_data()
absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)


In [3]:
# common imports
from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import Markdown, display

import pandas as pd
pd.set_option('display.max_colwidth', 0)
    
import numpy as np
import panel as pn

def printmd(string):
    display(Markdown(string))


# local imports
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

# specific imports
import math
import matplotlib.pyplot as plt
import seaborn as sb
# from sklearn.linear_model import LinearRegression
import os



## 2. Read the survey file and extract numeric variables

In [4]:
# read the csv file
df = panellibs.extract_data(absolutePath + csv_file)# print(absolutePath + csv_file)

# create a list of variable names
variables_df = pd.DataFrame({'varname':df.columns})
printmd("<b><span style='color:red'>All variables in the survey file:</span></b>")
print(variables_df.varname.values)

# create a dictionary of #number variables with abbreviated and full variable names 
var_list = {n[:n.index('#')]:n for n in variables_df.varname.values if '#number' in n}
printmd("<b><span style='color:red'>Numeric variables:</span></b>")
for key, value in var_list.items():
    print(key, ' : ', value)

#create a dataframe of only #number variables
nums_df = df[[n for n in variables_df.varname.values if '#number' in n]]


## 3. Matrix of scatterplots

- Select variables for the scatterplots in the first cell
- Generate the scatterplots in the second cell

In [5]:
# Coerce #number variables that aren't numeric, to be numeric, if possible
for i in list(nums_df):
    pd.to_numeric(i, errors='coerce')


In [6]:
# Examine the dataframe
with pd.option_context("display.max_columns", None):
    display(nums_df)


In [12]:
# 3.1 Choose variables for scattermatrix

#creating widgets
wlist = []
i = 0

while i < 5 and i < len(var_list):
    wlist.append(widgets.Dropdown(options=var_list.keys(), value = list(var_list.keys())[i], description = str(i+1) + ":"))
    i += 1
  
printmd("<b><span style='color:red'>Select variables for scatter matrix plotting, then run the next cell</span></b>")

for n in wlist:
    display(n)

In [16]:
# 3.2 Create the scattermatrix

#scattermatrix
scat_df = df[[var_list[n.value] for n in wlist]]

#check if all variables are different
if len(list(scat_df)) != scat_df.columns.nunique():
    printmd("<b><span style='color:red'>Make sure all selected variables are different, then rerun this cell</span></b>")
else:
    
    # keep only variables that are really numeric
    scat2_df = scat_df.select_dtypes(include=[np.number])
    for col in scat2_df.columns:
#         print(col+"   "+ str(scat2_df[col].nunique()))
        if scat2_df[col].nunique() < 2:
            scat2_df.drop(col, axis=1, inplace=True)

#     for col in scat_df.columns:
#         print(col+"   "+ str(scat_df[col].nunique()))

    # which columns were excluded:
    for item in list(set(list(scat_df.columns)) - set(scat2_df.columns)):
        printmd("<b><span style='color:red'>Non-numeric or singular variables excluded: </span></b>" + item)
        print(scat_df[item].describe())

    w2dict = {n[:n.index('#')]:n for n in scat2_df.columns}
    w2list = list(w2dict.keys())
    
    #plot
    printmd("<br><b><span style='color:red'>Scatter matrix for the selected numeric variables</span></b>")
    wot = pd.plotting.scatter_matrix(scat2_df, alpha=0.2, figsize=(10, 10), diagonal='kde')
    #axis
    for n in range(len(wot)):
        for j in range(len(wot)):
            ax = wot[n, j]
            ax.set(xlabel=w2list[j], ylabel=w2list[n])



## 4. Scatterplot and linear regression

- Select variables for the regression in the first cell
- Generate the plot in the second cell

In [ ]:
# 4.1 Choose the variables for the scatterplot

#creating widgets
a4 = widgets.Dropdown(options=var_list.keys(), value = list(var_list.keys())[0], description = "x:")
b4 = widgets.Dropdown(options=var_list.keys(), value = list(var_list.keys())[1], description = "y:")
ui = widgets.VBox([a4, b4])
def f4(a, b):
    return ((a, b))

printmd("<b><span style='color:red'>Select x and y variables for plotting, then run the next cell</span></b>")

formula = widgets.interactive_output(f4, {'a': a4, 'b': b4})

display(ui, formula)


In [ ]:
var_list[a4.value]

In [ ]:
# 4.2 Create the scatterplot and regression line

from pandas.api.types import is_numeric_dtype

x4 = df[var_list[a4.value]].tolist()
y4 = df[var_list[b4.value]].tolist()

if not is_numeric_dtype(df[var_list[a4.value]]):
    printmd("<b><span style='color:red'>"+ var_list[a4.value] + " is not numeric. Cannot plot.</span></b>")
    print(df[var_list[a4.value]].describe())

if not is_numeric_dtype(df[var_list[b4.value]]):
    printmd("<b><span style='color:red'>"+ var_list[b4.value] + " is not numeric. Cannot plot.</span></b>")
    print(df[var_list[b4.value]].describe())

if is_numeric_dtype(df[var_list[b4.value]]) and is_numeric_dtype(df[var_list[a4.value]]):

# check if variables different
    if a4.value == b4.value:
        printmd("<b><span style='color:red'>Make sure the selected variables are different, then rerun this cell</span></b>")

    else:
        try:
            printmd("<b><span style='color:red'><br>Scatterplot with regression line and confidence intervals:</span></b>")
            ax = sb.regplot(x=var_list[a4.value], y=var_list[b4.value], data=df, scatter_kws={"s": 20, "color" : "red", "alpha" : 0.2})
            #axis
            ax.set(xlabel=a4.value, ylabel=b4.value)
            plt.show()
        except:
            printmd("<b><span style='color:red'>Cannot plot. One or both variables contain non-numeric values!!</span></b>")



## 5. Mean, standard deviation, skew, etc...

- Select a variable in the first cell
- Generate statistics in the second cell

In [ ]:
# 5.2 Choose the variable to calculate descriptive statistics

# creating widgets
a5 = widgets.Dropdown(options=var_list.keys(), description = "Variable:")
ui = widgets.VBox([a5])
def f5(a):
    return (a)

printmd("<b><span style='color:red'><br>Select variable for descriptive statistics, then run the next cell</span></b>")

formula = widgets.interactive_output(f5, {'a': a5})

display(ui, formula)

In [ ]:
# 5.2 Show the descriptive statistics

try:
    # calculating descriptive stats
    var = df[var_list[a5.value]]
    vmean = var.mean()
    vsd = var.std()
    vskew = var.skew()
    vvar = var.var()

    # printing descriptive stats
    print("Mean of variable   : " + str(vmean))
    print("Standard deviation : " + str(vsd))
    print("Variance           : " + str(vvar))
    print("Skew               : " + str(vskew))

    # plot
    print("Histogram          :")
    ax = var.hist()
    # axis
    ax.set(xlabel=a5.value)
    noout = plt.axvline(vmean, color='red', linestyle='dashed', linewidth=2)
except:
    printmd(var_list[a5.value] +"<b><span style='color:red'> contains non-numeric values!! Cannot compute</span></b>")
    print(df[var_list[a5.value]].describe())


## 6. Generate a new statistical variable to add to SuAVE

- select a variable and an operation in the first cell
- edit variable name in the second cell
- compute the new variable in the third cell

In [ ]:
# 6.1 Select a variable for processing

a6 = widgets.Dropdown(options=var_list.keys())
b6 = widgets.Dropdown(options=['Number of SDs', 'Abs dist from mean'])
ui = widgets.VBox([a6, b6])
def f6(a, b):
    return ((a, b))

formula = widgets.interactive_output(f6, {'a': a6, 'b': b6})

printmd("<b><span style='color:red'><br>Select a variable and a derivative to create, then run the next cell</span></b>")

display(ui, formula)

In [ ]:
# 6.2 Give a name to the new variable

def f62(Var_Name):
    return Var_Name
if b6.value == "Abs dist from mean":
    newvar = interact(f62, Var_Name= "ADFM" + " on " + a6.value +'#number')
elif b6.value == "Number of SDs":
    newvar = interact(f62, Var_Name= "NSD" + " for " + a6.value +'#number')
    
printmd("<b><span style='color:red'>After defining variable name hit Enter, then run the next cell</span></b>")


In [ ]:
# 6.3 Compute the new variable and format it for SuAVE

try: 
    mean = df[var_list[a6.value]].mean()
    std = df[var_list[a6.value]].std()

    if b6.value == 'Abs dist from mean':
        df[newvar.widget.result] = [abs(i - mean) if not math.isnan(i) else np.nan for i in df[var_list[a6.value]]]
    elif b6.value == 'Number of SDs':
        df[newvar.widget.result] = [math.ceil(abs(i - mean) / std) if not math.isnan(i) else np.nan for i in df[var_list[a6.value]]]


    # make sure there are no illegal NaN type values in this #number variable
    df[newvar.widget.result] = df[newvar.widget.result].fillna('')
    df[newvar.widget.result] = pd.to_numeric(df[newvar.widget.result], errors='coerce', downcast='float')
    df[newvar.widget.result] = df[newvar.widget.result].apply(lambda x: '{:.6f}'.format(x))
    df.replace(['None', 'nan'], np.nan, inplace=True)
    printmd("<b><span style='color:red'>New variable computed</span></b>")

except:
    printmd(var_list[a6.value] +"<b><span style='color:red'> contains non-numeric values!! Cannot compute</span></b>")
    print(df[var_list[a6.value]].describe())


## 7. Visualize the resultant dataframe

In [ ]:
with pd.option_context("display.max_columns", None):
    if any("geometry" in col for col in df.columns):
        display(df.drop(['geometry'],axis=1))
    else:
        display(df)
    
    

## 8. Save the new version of CSV file, and give a name to new survey

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)


## 9. Generate the survey and create survey URL

In [ ]:
suaveint.create_survey(survey_url,new_file, survey_name, dzc_file, user, csv_file, view, views)